# CryoEM Particle Picker - Google Colab Testing Notebook

This notebook allows you to test the CryoEM Particle Picker tool with real datasets in Google Colab.

## Features:
- ✅ Test with real CryoEM datasets
- ✅ Support for .mrc, .mrcs, .st file formats
- ✅ Automatic visualization generation
- ✅ Download results (STAR files + visualizations)
- ✅ GPU acceleration (if available)

## Workflow:
1. Install dependencies
2. Clone/upload the tool
3. Upload your CryoEM data OR use sample data
4. Run particle picking
5. View and download results

## Step 1: Check GPU Availability

In [ ]:
import torch
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = 'cuda'
else:
    print("Running on CPU")
    device = 'cpu'

## Step 2: Install Dependencies

In [ ]:
%%capture
# Install required packages
!pip install ultralytics mrcfile numpy scipy matplotlib seaborn plotly numba pyyaml pillow opencv-python

## Step 3: Clone the Tool from GitHub

In [ ]:
import os
from pathlib import Path

# Define project paths
PROJECT_NAME = 'cryoem-precision-tool'
REPO_URL = 'https://github.com/jatin_bioinformatics/cryoem-precision-tool.git'  # Replace with your URL

# Set up directory structure
BASE_DIR = Path('/content')
PROJECT_DIR = BASE_DIR / PROJECT_NAME
TOOL_DIR = PROJECT_DIR / 'cryoEM_particle_picker'
LIB_DIR = TOOL_DIR / 'lib'
DATA_DIR = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'

# Clone or update repository
if not PROJECT_DIR.exists():
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repository already exists. Pulling latest changes...")
    !cd {PROJECT_DIR} && git pull

# Change to tool directory
os.chdir(str(TOOL_DIR))
print(f"✅ Current directory: {os.getcwd()}")
print(f"✅ Tool directory: {TOOL_DIR}")
print(f"✅ Lib directory: {LIB_DIR}")

# Verify directory structure
if LIB_DIR.exists():
    print(f"✅ Found lib directory with modules: {list(LIB_DIR.glob('*/'))}")
else:
    print(f"❌ Lib directory not found at {LIB_DIR}")

## Step 4: Set Up Python Path

In [ ]:
import sys
from pathlib import Path

# Add lib directory to Python path
sys.path.insert(0, str(LIB_DIR))
print(f"✅ Added to Python path: {LIB_DIR}")

# Verify imports work
try:
    from preprocessing import load_micrograph_mmap, normalize_micrograph
    from ml_model.picker import load_model_cached
    from ml_model.Inference import FastInferenceEngine
    from postprocessing import filter_by_confidence, non_maximum_suppression, apply_edge_exclusion
    from ctf import CTFEstimator
    from utils import write_star_file, write_confidence_map, generate_statistics_plot
    print("✅ All imports successful!")
    
    # Show available modules
    modules = [d.name for d in LIB_DIR.iterdir() if d.is_dir() and not d.name.startswith('__')]
    print(f"✅ Available modules: {modules}")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print(f"Current Python path: {sys.path[:3]}...")  # Show first 3 entries
    print(f"Lib directory contents: {list(LIB_DIR.glob('*'))}")

## Step 5: Upload Your CryoEM Data

### Option A: Upload from your computer

In [ ]:
from google.colab import files
import os

# Create data directory
DATA_DIR.mkdir(exist_ok=True)
os.chdir(str(DATA_DIR))

print(f"📁 Upload directory: {DATA_DIR}")
print("Upload your CryoEM file (.mrc, .mrcs, or .st):")
uploaded = files.upload()

# Get uploaded filename and full path
filename = list(uploaded.keys())[0]
input_file = DATA_DIR / filename

print(f"\n✅ Uploaded: {filename}")
print(f"✅ Full path: {input_file}")
print(f"✅ File size: {input_file.stat().st_size / (1024*1024):.2f} MB")
print(f"✅ File exists: {input_file.exists()}")

# Change back to tool directory
os.chdir(str(TOOL_DIR))

### Option B: Download sample data from EMPIAR

In [ ]:
# Download a sample micrograph from EMPIAR (Example: EMPIAR-10025)
# This is a small test file (~50MB)

import urllib.request
import os

os.makedirs('/content/data', exist_ok=True)
os.chdir('/content/data')

# Sample CryoEM micrograph URL (replace with actual EMPIAR URL)
sample_url = "https://ftp.ebi.ac.uk/empiar/world_availability/10025/data/micrographs/sample_micrograph.mrc"

print("Downloading sample CryoEM data...")
# Uncomment to download:
# urllib.request.urlretrieve(sample_url, 'sample_micrograph.mrc')
# input_file = 'sample_micrograph.mrc'
# print(f"✅ Downloaded: {input_file}")

# Or use the test data from the repository
input_file = '/content/cryoem-precision-tool/cryoEM_particle_picker/archive/12128/data/20240114_JN_MB04_LO_004_Tomo_009.st'
if os.path.exists(input_file):
    print(f"✅ Using test data: {input_file}")
    print(f"File size: {os.path.getsize(input_file) / (1024*1024):.2f} MB")
else:
    print("❌ Test data not found. Please upload your own file using Option A.")

## Step 6: Configure Particle Picking Parameters

In [ ]:
# Configure parameters
params = {
    'input': input_file,
    'output': '/content/results/particles.star',
    'particle_size': 200,  # Particle diameter in pixels (adjust for your data)
    'confidence_threshold': 0.7,  # 0.1-0.99 (higher = fewer but more accurate picks)
    'device': device,  # 'cuda' or 'cpu'
    'batch_size': 128,
    'min_distance': None,  # Minimum distance between particles (default: particle_size)
    'edge_exclusion': 50,  # Exclude particles within N pixels of edge
    'ctf_estimation': True,  # Enable CTF parameter estimation
    'voltage': 300.0,  # kV
    'cs': 2.7,  # mm
    'pixel_size': 1.0,  # Angstroms/pixel
    'output_heatmap': '/content/results/heatmap.png',
    'output_statistics': '/content/results/statistics.png',
    'use_plotly': False,  # Use matplotlib (plotly requires kaleido)
}

# Create output directory
os.makedirs('/content/results', exist_ok=True)

print("Particle Picking Configuration:")
print("="*50)
for key, value in params.items():
    print(f"{key:25s}: {value}")
print("="*50)

## Step 7: Run Particle Picking

In [ ]:
import time
import numpy as np
from pathlib import Path

# Import modules
from preprocessing import load_micrograph_mmap, normalize_micrograph
from ml_model.picker import load_model_cached
from ml_model.Inference import FastInferenceEngine
from postprocessing import filter_by_confidence, non_maximum_suppression, apply_edge_exclusion
from ctf import CTFEstimator
from utils import write_star_file, write_confidence_map, generate_statistics_plot

print("\n" + "="*60)
print("Starting Particle Picking...")
print("="*60 + "\n")

start_time = time.time()

# Step 1: Load micrograph
print("[1/8] Loading micrograph...")
micrograph = load_micrograph_mmap(params['input'])
print(f"      Shape: {micrograph.shape}")
print(f"      Memory: {micrograph.nbytes / (1024*1024):.2f} MB")

# Step 2: Normalize
print("\n[2/8] Normalizing micrograph...")
micrograph_norm = normalize_micrograph(micrograph)
print(f"      Mean: {micrograph_norm.mean():.6f}")
print(f"      Std: {micrograph_norm.std():.6f}")

# Step 3: Load model
print("\n[3/8] Loading YOLOv8n model...")
model = load_model_cached('yolov8n.pt', device=params['device'])
print(f"      Device: {params['device']}")

# Step 4: Initialize inference engine
print("\n[4/8] Initializing inference engine...")
engine = FastInferenceEngine(model, device=params['device'], use_amp=(params['device']=='cuda'))
engine.batch_size = params['batch_size']

# Step 5: Run detection
print("\n[5/8] Running particle detection...")
coords, confidences = engine.predict_micrograph(
    micrograph_norm,
    particle_size=params['particle_size'],
    conf_threshold=params['confidence_threshold'] * 0.5
)
print(f"      Initial detections: {len(coords)} particles")

# Step 6: Apply filters
print("\n[6/8] Applying post-processing filters...")

# Confidence filtering
coords, confidences = filter_by_confidence(coords, confidences, params['confidence_threshold'])
print(f"      After confidence filter: {len(coords)} particles")

# Non-maximum suppression
min_dist = params['min_distance'] if params['min_distance'] else params['particle_size']
coords, confidences = non_maximum_suppression(coords, confidences, min_dist)
print(f"      After NMS: {len(coords)} particles")

# Edge exclusion
if params['edge_exclusion'] > 0:
    coords, confidences = apply_edge_exclusion(
        coords, confidences, micrograph.shape, params['edge_exclusion']
    )
    print(f"      After edge exclusion: {len(coords)} particles")

# Step 7: CTF estimation (optional)
ctf_params = None
if params['ctf_estimation']:
    print("\n[7/8] Estimating CTF parameters...")
    ctf_estimator = CTFEstimator(
        voltage=params['voltage'],
        cs=params['cs'],
        pixel_size=params['pixel_size']
    )
    ctf_params = ctf_estimator.estimate(micrograph)
    print(f"      Defocus: {ctf_params.average_defocus():.2f} μm")
    print(f"      Astigmatism: {ctf_params.astigmatism:.2f} Å")
    print(f"      Fit resolution: {ctf_params.fit_resolution:.2f} Å")
else:
    print("\n[7/8] Skipping CTF estimation...")

# Step 8: Write outputs
print("\n[8/8] Writing outputs...")

# STAR file
write_star_file(
    params['output'],
    coords,
    confidences,
    ctf_params=ctf_params,
    micrograph_name=Path(params['input']).name
)
print(f"      ✅ STAR file: {params['output']}")

# Heatmap
write_confidence_map(
    params['output_heatmap'],
    micrograph.shape,
    coords,
    confidences,
    use_plotly=params['use_plotly']
)
print(f"      ✅ Heatmap: {params['output_heatmap']}")

# Statistics
generate_statistics_plot(coords, confidences, params['output_statistics'])
print(f"      ✅ Statistics: {params['output_statistics']}")

elapsed = time.time() - start_time

print("\n" + "="*60)
print("✅ Particle Picking Complete!")
print("="*60)
print(f"Particles detected: {len(coords)}")
print(f"Mean confidence: {confidences.mean():.3f}")
print(f"Std confidence: {confidences.std():.3f}")
print(f"Processing time: {elapsed:.2f}s")
print(f"Speed: {elapsed/len(coords):.3f}s per particle" if len(coords) > 0 else "No particles detected")
print("="*60)

## Step 8: View Results

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt

print("Particle Picks Visualization:")
print("="*60)
display(Image(filename=params['output_heatmap']))

print("\nDetection Statistics:")
print("="*60)
display(Image(filename=params['output_statistics']))

## Step 9: Inspect STAR File

In [ ]:
# Read and display first 10 lines of STAR file
with open(params['output'], 'r') as f:
    lines = f.readlines()
    print("STAR File Contents (first 20 lines):")
    print("="*60)
    for i, line in enumerate(lines[:20]):
        print(f"{i+1:3d}: {line}", end='')
    if len(lines) > 20:
        print(f"\n... ({len(lines)-20} more lines)")

print(f"\nTotal particles in STAR file: {len(lines) - len([l for l in lines if l.startswith('_') or l.startswith('data_') or l.startswith('loop_') or l.strip() == ''])}")

## Step 10: Download Results

In [ ]:
from google.colab import files
import zipfile

# Create a zip file with all results
zip_filename = '/content/cryoem_results.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    zipf.write(params['output'], 'particles.star')
    zipf.write(params['output_heatmap'], 'heatmap.png')
    zipf.write(params['output_statistics'], 'statistics.png')

print("Downloading results...")
files.download(zip_filename)
print("✅ Downloaded: cryoem_results.zip")
print("\nContents:")
print("  - particles.star (coordinates)")
print("  - heatmap.png (particle picks visualization)")
print("  - statistics.png (detection statistics)")

## Step 11: Batch Processing (Optional)

Process multiple micrographs at once

In [ ]:
# Upload multiple files or specify a directory
import glob

# Example: Process all .mrc files in a directory
input_dir = '/content/data'
output_dir = '/content/batch_results'
os.makedirs(output_dir, exist_ok=True)

# Find all CryoEM files
files_to_process = []
for ext in ['*.mrc', '*.mrcs', '*.st']:
    files_to_process.extend(glob.glob(os.path.join(input_dir, ext)))

print(f"Found {len(files_to_process)} files to process")

if len(files_to_process) > 0:
    print("\nProcessing files:")
    for i, input_file in enumerate(files_to_process, 1):
        print(f"\n[{i}/{len(files_to_process)}] Processing: {os.path.basename(input_file)}")
        
        # Update output paths
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        params['input'] = input_file
        params['output'] = os.path.join(output_dir, f"{base_name}_particles.star")
        params['output_heatmap'] = os.path.join(output_dir, f"{base_name}_heatmap.png")
        params['output_statistics'] = os.path.join(output_dir, f"{base_name}_statistics.png")
        
        # Run particle picking (reuse code from Step 7)
        # ... (copy the processing code here)
        
    print(f"\n✅ Batch processing complete! Processed {len(files_to_process)} files.")
    print(f"Results saved to: {output_dir}")
else:
    print("No files found for batch processing.")

## Troubleshooting

### Common Issues:

1. **Out of Memory Error**
   - Reduce `batch_size` parameter
   - Use CPU instead of GPU
   - Process smaller images

2. **No Particles Detected**
   - Lower `confidence_threshold` (try 0.5 or 0.3)
   - Adjust `particle_size` parameter
   - Check if micrograph quality is good

3. **Too Many False Positives**
   - Increase `confidence_threshold` (try 0.8 or 0.9)
   - Increase `min_distance` parameter
   - Enable `edge_exclusion`

4. **Import Errors**
   - Restart runtime and re-run all cells
   - Check that all dependencies are installed
   - Verify Python path is set correctly

### Tips for Best Results:

- Start with default parameters and adjust based on results
- Use GPU for faster processing (Runtime → Change runtime type → GPU)
- For large datasets, process in batches
- Compare results with manual picks to optimize parameters
- Check visualizations to assess picking quality

## Next Steps

After particle picking, you can:

1. **Import to RELION**:
   - Download the STAR file
   - Import in RELION: `Particle extraction` → `Input coordinates`
   - Continue with 2D classification

2. **Import to CryoSPARC**:
   - Download the STAR file
   - Import in CryoSPARC: `Import Particle Stack`
   - Continue with 2D classification or Ab-initio reconstruction

3. **Fine-tune the Model** (Advanced):
   - Collect training data with manual picks
   - Fine-tune YOLOv8n on your specific particle type
   - Improve detection accuracy

4. **Batch Process Large Datasets**:
   - Use the batch processing code above
   - Process entire data collection sessions
   - Generate summary reports

## Advanced: Using More Powerful Models

The tool currently uses YOLOv8n (6.2M parameters). You can upgrade to more powerful models for better accuracy.

### Option 1: Larger YOLO Models

In [ ]:
# Test different YOLO model sizes
model_options = {
    'yolov8n.pt': {'params': '6.2M', 'speed': 'Fastest', 'accuracy': 'Good'},
    'yolov8s.pt': {'params': '11.2M', 'speed': 'Fast', 'accuracy': 'Better'},
    'yolov8m.pt': {'params': '25.9M', 'speed': 'Medium', 'accuracy': 'Much Better'},
    'yolov8l.pt': {'params': '43.7M', 'speed': 'Slow', 'accuracy': 'Excellent'},
    'yolov8x.pt': {'params': '68.2M', 'speed': 'Slowest', 'accuracy': 'Best'}
}

print("Available YOLO Models:")
print("=" * 60)
for model, specs in model_options.items():
    print(f"{model:12s} | {specs['params']:>8s} | {specs['speed']:>8s} | {specs['accuracy']}")

# Choose a more powerful model
selected_model = 'yolov8m.pt'  # Change this to test different models
print(f"\n✅ Selected model: {selected_model}")
print(f"   Parameters: {model_options[selected_model]['params']}")
print(f"   Expected accuracy: {model_options[selected_model]['accuracy']}")

In [ ]:
# Load the more powerful model
print(f"Loading {selected_model}...")
powerful_model = load_model_cached(selected_model, device=device)
print(f"✅ Loaded {selected_model} on {device}")

# Create new inference engine
powerful_engine = FastInferenceEngine(powerful_model, device=device, use_amp=(device=='cuda'))
powerful_engine.batch_size = params['batch_size'] // 2  # Reduce batch size for larger models

print(f"✅ Inference engine ready with batch size {powerful_engine.batch_size}")

In [ ]:
# Compare results between models
print("Comparing YOLOv8n vs More Powerful Model:")
print("=" * 60)

# Run with powerful model
start_time = time.time()
coords_powerful, confidences_powerful = powerful_engine.predict_micrograph(
    micrograph_norm,
    particle_size=params['particle_size'],
    conf_threshold=params['confidence_threshold'] * 0.5
)
powerful_time = time.time() - start_time

# Apply same post-processing
coords_powerful, confidences_powerful = filter_by_confidence(
    coords_powerful, confidences_powerful, params['confidence_threshold']
)
coords_powerful, confidences_powerful = non_maximum_suppression(
    coords_powerful, confidences_powerful, params['particle_size']
)
if params['edge_exclusion'] > 0:
    coords_powerful, confidences_powerful = apply_edge_exclusion(
        coords_powerful, confidences_powerful, micrograph.shape, params['edge_exclusion']
    )

# Compare results
print(f"YOLOv8n Results:")
print(f"  Particles: {len(coords)}")
print(f"  Mean confidence: {confidences.mean():.3f}")
print(f"  Processing time: {elapsed:.2f}s")

print(f"\n{selected_model} Results:")
print(f"  Particles: {len(coords_powerful)}")
print(f"  Mean confidence: {confidences_powerful.mean():.3f}")
print(f"  Processing time: {powerful_time:.2f}s")

print(f"\nImprovement:")
print(f"  Particle count change: {len(coords_powerful) - len(coords):+d}")
print(f"  Confidence improvement: {confidences_powerful.mean() - confidences.mean():+.3f}")
print(f"  Speed ratio: {powerful_time / elapsed:.2f}x slower")

### Option 2: Fine-tuned CryoEM Model (Advanced)

For best results, you can fine-tune a model on your specific particle type:

In [ ]:
# This is a demonstration of how to fine-tune a model
# In practice, you would need labeled training data

print("Fine-tuning Process (Demonstration):")
print("=" * 50)
print("1. Collect manual picks from experts (100-1000 micrographs)")
print("2. Convert RELION/CryoSPARC coordinates to YOLO format")
print("3. Split data: 80% training, 20% validation")
print("4. Fine-tune pretrained model:")
print("")
print("   from ultralytics import YOLO")
print("   model = YOLO('yolov8m.pt')")
print("   results = model.train(")
print("       data='cryoem_dataset.yaml',")
print("       epochs=100,")
print("       imgsz=640,")
print("       batch=16")
print("   )")
print("")
print("5. Save fine-tuned model as 'yolov8m_cryoem.pt'")
print("6. Tool automatically uses fine-tuned model if available")
print("")
print("Expected improvements:")
print("  - 20-50% better accuracy on your particle type")
print("  - Fewer false positives")
print("  - Better handling of particle orientations")
print("  - Improved performance on similar datasets")

### Option 3: Alternative Model Architectures

For research applications, you can integrate other model types:

In [ ]:
# Example: Using DETR (Detection Transformer)
# Uncomment to install and test:

# !pip install transformers

# from transformers import DetrImageProcessor, DetrForObjectDetection
# import torch
# from PIL import Image

# # Load DETR model
# processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
# detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")

# # Convert micrograph to PIL Image
# micrograph_pil = Image.fromarray(micrograph_norm).convert("RGB")

# # Run DETR inference
# inputs = processor(images=micrograph_pil, return_tensors="pt")
# outputs = detr_model(**inputs)

# # Post-process
# target_sizes = torch.tensor([micrograph_pil.size[::-1]])
# results = processor.post_process_object_detection(
#     outputs, target_sizes=target_sizes, threshold=0.7
# )[0]

# print(f"DETR detected {len(results['boxes'])} objects")

print("Alternative Model Architectures:")
print("=" * 40)
print("1. DETR (Detection Transformer):")
print("   - Transformer-based architecture")
print("   - End-to-end object detection")
print("   - Good for complex scenes")
print("")
print("2. Faster R-CNN:")
print("   - Two-stage detection")
print("   - High accuracy")
print("   - Slower than YOLO")
print("")
print("3. CryoEM-specific tools:")
print("   - crYOLO: Specialized for CryoEM")
print("   - Topaz: Deep learning particle picker")
print("   - EMAN2: Traditional + ML methods")
print("")
print("See ML_MODEL_INTEGRATION_GUIDE.md for implementation details")

### Model Selection Recommendations

**For different use cases:**

1. **Quick Testing**: YOLOv8n (current default)
   - Fast processing
   - Good for initial screening
   - Low GPU memory requirements

2. **Production Use**: YOLOv8m or YOLOv8l
   - Better accuracy
   - Still reasonably fast
   - Good balance of speed/accuracy

3. **High-Quality Results**: YOLOv8x + Fine-tuning
   - Best accuracy
   - Requires more computational resources
   - Ideal for final publication results

4. **Research/Comparison**: Multiple models
   - Compare YOLO vs DETR vs Faster R-CNN
   - Ensemble methods
   - Custom architectures